In [54]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import time
import re
from urllib.parse import urljoin, urlparse
import pydantic

import ollama
from ollama import chat
from openai import OpenAI
import os


In [56]:
from typing import Any, Dict, List, Literal, Optional, Union
from pydantic import BaseModel, Field, ConfigDict

# ---------- Atomic / reusable models ----------

class Evidence(BaseModel):
    quote: Optional[str] = None
    confidence: float = Field(default=0.0, ge=0.0, le=1.0)
    where: Optional[str] = None


class PowerTarget(BaseModel):
    value: Optional[Union[int, float]] = None
    unit: Literal["whp"] = "whp"
    notes: Optional[str] = None


class Boost(BaseModel):
    psi: Optional[Union[int, float]] = None
    notes: Optional[str] = None


class Money(BaseModel):
    value: Optional[Union[int, float]] = None
    currency: Literal["USD"] = "USD"
    notes: Optional[str] = None


# ---------- Top-level sections ----------

class Source(BaseModel):
    platform: Literal["supramkv"] = "supramkv"
    thread_url: Optional[str] = None
    thread_title: Optional[str] = None


class ThreadSummary(BaseModel):
    one_sentence: Optional[str] = None
    key_takeaways: List[str] = Field(default_factory=list)


class Fueling(BaseModel):
    fuel_type: str = "unknown"  # keep flexible; default matches schema
    injectors: Optional[str] = None
    pump: Optional[str] = None
    hpfp: Optional[str] = None
    port_injection: Optional[str] = None
    notes: Optional[str] = None


class Hardware(BaseModel):
    turbo: Optional[str] = None
    manifold: Optional[str] = None
    downpipe: Optional[str] = None
    intake: Optional[str] = None
    intercooler: Optional[str] = None
    exhaust: Optional[str] = None
    spark_plugs: Optional[str] = None
    cooling: List[str] = Field(default_factory=list)
    drivetrain: List[str] = Field(default_factory=list)
    engine_internal: List[str] = Field(default_factory=list)
    notes: Optional[str] = None


class Tuning(BaseModel):
    tuner_or_shop: Optional[str] = None
    ecu_solution: Optional[str] = None
    maps: List[str] = Field(default_factory=list)
    notes: Optional[str] = None


class Route(BaseModel):
    route_name: Optional[str] = None
    route_type: Optional[str] = None
    power_targets: List[PowerTarget] = Field(default_factory=lambda: [PowerTarget()])
    fueling: Fueling = Field(default_factory=Fueling)
    boost: Boost = Field(default_factory=Boost)
    hardware: Hardware = Field(default_factory=Hardware)
    tuning: Tuning = Field(default_factory=Tuning)
    supporting_mods: List[str] = Field(default_factory=list)
    risks_and_limits: List[str] = Field(default_factory=list)
    estimated_cost: Money = Field(default_factory=Money)
    evidence: List[Evidence] = Field(default_factory=lambda: [Evidence()])


class Part(BaseModel):
    part_name: Optional[str] = None
    category: Optional[str] = None
    brand: Optional[str] = None
    model: Optional[str] = None
    specs: Dict[str, Any] = Field(default_factory=dict)
    mentioned_reason: Optional[str] = None


class NamedEntities(BaseModel):
    brands: List[str] = Field(default_factory=list)
    shops_or_tuners: List[str] = Field(default_factory=list)
    platforms_or_ecu: List[str] = Field(default_factory=list)
    fuels: List[str] = Field(default_factory=list)
    power_numbers: List[str] = Field(default_factory=list)


class Quality(BaseModel):
    extraction_notes: List[str] = Field(default_factory=list)
    missing_critical_fields: List[str] = Field(default_factory=list)


class SupraThreadExtraction(BaseModel):
    """
    Enforces the exact top-level shape you specified.

    - extra="forbid" prevents unexpected keys (useful if you truly want exact shape)
    - defaults mirror your schema (nulls and empty lists)
    """
    model_config = ConfigDict(extra="forbid")

    source: Source = Field(default_factory=Source)
    thread_summary: ThreadSummary = Field(default_factory=ThreadSummary)
    routes: List[Route] = Field(default_factory=lambda: [Route()])
    parts: List[Part] = Field(default_factory=list, min_length=1)
    named_entities: NamedEntities = Field(default_factory=NamedEntities)
    quality: Quality = Field(default_factory=Quality)

In [22]:
hp_academy_data_directory = "../hp_acadamy_scrapper/data"
forum_data_directory = "../redline_forum_scrapper/data/forum_data"


In [23]:
#load data from hp academy and forum data directories
hp_academy_data = pd.read_csv(os.path.join(hp_academy_data_directory, "hpacademy_course_content.csv"))
forum_data = pd.read_csv(os.path.join(forum_data_directory, "power_guide_csv.csv"))


In [26]:
forum_data['thread_url'] = "https://www.supramkv.com/threads/gr-supra-101-routes-to-big-power.17670/" + forum_data['post_id'].astype(str)

In [27]:
forum_data.head()

,author,content,date,post_id,links,images,youtube_videos,thread_url
0,D3ad_Hand,Since the forums are ever more active and new ...,2023-04-29T02:16:05-0700,post-276037,"['https://www.supramkv.com/members/4250/', 'ht...",[],[],https://www.supramkv.com/threads/gr-supra-101-...
1,jtsang25,If this doesn't get pinned it's going to get l...,2023-04-29T03:22:23-0700,post-276041,[],[],[],https://www.supramkv.com/threads/gr-supra-101-...
2,D3ad_Hand,jtsang25 said:\nIf this doesn't get pinned it'...,2023-04-29T03:36:03-0700,post-276042,['https://www.supramkv.com/goto/post?id=276041'],[],[],https://www.supramkv.com/threads/gr-supra-101-...
3,DSG Performance,Very thorough and comprehensive! Fantastic,2023-04-29T04:52:08-0700,post-276043,[],[],[],https://www.supramkv.com/threads/gr-supra-101-...
4,Basura,"Bookmarked just in case, great info!",2023-04-29T04:57:08-0700,post-276044,[],[],[],https://www.supramkv.com/threads/gr-supra-101-...


In [ ]:
# Ollama Constants
# Local deepseek-r1:8b model
Ollama_api = "http://localhost:11434/api/chat"

Headers = {"Content-Type": "application/json"}

Model = "deepseek-r1:8b"

In [63]:
system_prompt = """
You are a precision information extraction engine. Your job is to convert an automotive forum post into a STRICT JSON object that matches the provided schema exactly.
Rules:
- Output MUST be valid JSON only. No markdown. No commentary.
- Preserve the schema shape exactly (all top-level keys must exist).
- If a field is unknown, use null (or the schema default like "unknown" for fuel_type, "whp" for unit, "USD" for currency).
- Do not invent brands, parts, numbers, or claims. Only extract what is explicitly supported by the text.
- Create MULTIPLE routes when the post describes multiple power levels, stages, options, or “to reach X hp do Y” paths.
- Associate evidence: for every route, include 1–5 evidence objects with short quotes (<= 25 words each) taken verbatim from the post, and set confidence 0.0–1.0 based on clarity.
- De-duplicate repeated information. Prefer the clearest mention.
- If the post implies “700whp = 500whp mods + additional upgrades”, put that relationship in route.hardware.notes and/or route.tuning.notes.
"""

user_prompt_template = """
You will be given:
1) A JSON schema template that defines the required output shape.
2) A forum post (plain text and/or HTML).

TASK:
Extract structured data from the forum post into the schema.

EXTRACTION PROCEDURE (follow exactly):
1) Route discovery & segmentation:
   - Identify distinct "routes" (power levels, stages, options). A route is any coherent path to a power target (e.g., 500whp vs 700whp).
   - If multiple power targets exist, create one Route per target or per clearly distinct option.
2) Populate each route:
   - route_name: concise label (e.g., "500whp pump gas", "700whp E85 + PI").
   - route_type: if present (e.g., "bolt-ons", "big turbo", "hybrid turbo", "single turbo", "stage 2"). Otherwise null.
   - power_targets: include every explicit target number, using unit "whp" unless post clearly uses hp; if hp only, still store numeric in value but note "hp" in notes.
   - fueling: extract fuel type(s) (E85, pump gas, race gas, meth, etc). If not stated, keep "unknown".
   - boost.psi: only if explicitly stated.
   - hardware: extract parts and upgrades. Use arrays for cooling/drivetrain/engine_internal; keep other fields as strings.
   - tuning: tuner/shop, ECU platform, maps.
   - supporting_mods: other mods not fitting above (tires, brakes, suspension, etc.) if relevant to power goal.
   - risks_and_limits: any reliability, heat, fueling limits, transmission limits, etc.
   - estimated_cost: only if explicit numbers/currency appear. Otherwise null value.
   - evidence: quote + where (e.g., "Post #12", "Paragraph 3", "line 45") if you can infer; otherwise null.
3) Global parts list:
   - Create parts[] for notable parts mentioned anywhere in the post.
   - Each part should be a single object with category/brand/model when possible.
   - Put technical values into specs (e.g., {{"injector_cc": 1050, "turbine_housing": "0.72 A/R"}}).
   - mentioned_reason: why it was recommended (from text).
4) Named entities:
   - brands: brand names
   - shops_or_tuners: tuner/shop names
   - platforms_or_ecu: "BM3", "MHD", "ECUTEK", "Motec", etc.
   - fuels: unique fuel types found
   - power_numbers: strings like "500whp", "700hp", "30psi"
5) Quality:
   - extraction_notes: any ambiguity you encountered (e.g., "Power target mentioned without fuel type")
   - missing_critical_fields: list of dot-paths for key items that are missing for at least one route (e.g., "routes[0].hardware.turbo", "routes[1].fueling.fuel_type")

OUTPUT:
Return ONLY the JSON object matching the schema.

FORUM POST:
{forum_post_content}
"""

prompt_template = """
You are an information extraction engine. Your job is to read a single forum thread page (HTML or text) and return ONLY valid JSON that matches the schema below.
RETURN IN the SCHEMA BELOW IN JSON ONLY. DO NOT RETURN ANYTHING ELSE.
RULES
- Output MUST be a single JSON object. No markdown, no commentary, no backticks.
- If a field is unknown, use null (not empty string).
- Do not hallucinate. Only extract what is present in the input.
- Prefer exact numbers + units when available (e.g., "700 whp", "93 octane", "E85", "psi", "nm", "lb-ft").
- Normalize common entities:
  - fuel_type: one of ["pump_gas", "e85", "race_gas", "meth", "diesel", "unknown"]
  - transmission: one of ["auto", "manual", "dct", "unknown"]
  - build_goal_type: one of ["whp_target", "trap_speed", "lap_time", "reliability", "street", "track", "drag", "mixed", "unknown"]
- A "route" means a distinct pathway to big power (e.g., turbo kit A vs B, stock turbo upgrade vs single turbo, fueling path, supporting mods path).
- Extract routes even if they are implied by headings, bullets, or repeated recommendations.

SCHEMA (return exactly this shape)
{{
  "source": {{
    "platform": "supramkv",
    "thread_url": null,
    "thread_title": null
  }},
  "thread_summary": {{
    "one_sentence": null,
    "key_takeaways": []
  }},
  "routes": [
    {{
      "route_name": null,
      "route_type": null,
      "power_targets": [
        {{ "value": null, "unit": "whp", "notes": null }}
      ],
      "fueling": {{
        "fuel_type": "unknown",
        "injectors": null,
        "pump": null,
        "hpfp": null,
        "port_injection": null,
        "notes": null
      }},
      "boost": {{ "psi": null, "notes": null }},
      "hardware": {{
        "turbo": null,
        "manifold": null,
        "downpipe": null,
        "intake": null,
        "intercooler": null,
        "exhaust": null,
        "spark_plugs": null,
        "cooling": [],
        "drivetrain": [],
        "engine_internal": [],
        "notes": null
      }},
      "tuning": {{
        "tuner_or_shop": null,
        "ecu_solution": null,
        "maps": [],
        "notes": null
      }},
      "supporting_mods": [],
      "risks_and_limits": [],
      "estimated_cost": {{ "value": null, "currency": "USD", "notes": null }},
      "evidence": [
        {{ "quote": null, "confidence": 0.0, "where": null }}
      ]
    }}
  ],
  "parts": [
    {{
      "part_name": null,
      "category": null,
      "brand": null,
      "model": null,
      "specs": {{}},
      "mentioned_reason": null
    }}
  ],
  "named_entities": {{
    "brands": [],
    "shops_or_tuners": [],
    "platforms_or_ecu": [],
    "fuels": [],
    "power_numbers": []
  }},
  "quality": {{
    "extraction_notes": [],
    "missing_critical_fields": []
  }}
}}

EXTRACTION STEPS
1) Identify thread_title and any high-level summary.
2) Find all distinct power routes. For each route: extract target WHP, fueling approach, turbo/hardware, tuning/ECU approach, reliability caveats, and any cost mentions.
3) Extract parts into the "parts" list (dedupe by (brand, model, category) if possible).
4) Fill "named_entities" lists.
5) Add "evidence" quotes for each route: 1–3 short direct snippets from the input that justify the extraction (<= 25 words per quote).
6) Return final JSON only.

INPUT:
{THREAD_CONTENT}
"""

thread_url = forum_data.loc[0, "thread_url"]
thread_content = forum_data.loc[0, "content"]

final_prompt = prompt_template.format(THREAD_CONTENT=thread_content)


In [64]:
thread_url = forum_data.loc[0, "thread_url"]
thread_content = forum_data.loc[0, "content"]

final_prompt = user_prompt_template.format(forum_post_content=thread_content)

In [65]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": final_prompt}
]

response = chat(
    model=Model,
    messages=messages, 
    format = SupraThreadExtraction.model_json_schema(),
    stream=False,
)

supra_data = SupraThreadExtraction.model_validate_json(response.message.content)
print(supra_data)

source=Source(platform='supramkv', thread_url=None, thread_title=None) thread_summary=ThreadSummary(one_sentence=None, key_takeaways=[]) routes=[Route(route_name='500whp', route_type=None, power_targets=[PowerTarget(value=None, unit='whp', notes='Can be achieved with bolt-on modifications and ECU tuning alone. Requires proper fueling (flex fuel kit recommended for E85/E-blend). Note: Ensure fuel system supports power level (e.g., high-pressure fuel pump).'), PowerTarget(value=None, unit='whp', notes='Bolt-on mods include CAI, downpipe, headers, TB spacer, ECU remap/tune. Quality parts essential for reliability. Note: E85/E-blend requires flex fuel kit for optimal performance and protection.')], fueling=Fueling(fuel_type='unknown', injectors=None, pump=None, hpfp=None, port_injection=None, notes=None), boost=Boost(psi=None, notes=None), hardware=Hardware(turbo=None, manifold=None, downpipe=None, intake=None, intercooler=None, exhaust=None, spark_plugs=None, cooling=[], drivetrain=[], en

In [71]:
#display json in pandas dataframe
supra_data_df = pd.json_normalize(supra_data.model_dump())
print(supra_data_df['routes'].values.tolist())

[[{'route_name': '500whp', 'route_type': None, 'power_targets': [{'value': None, 'unit': 'whp', 'notes': 'Can be achieved with bolt-on modifications and ECU tuning alone. Requires proper fueling (flex fuel kit recommended for E85/E-blend). Note: Ensure fuel system supports power level (e.g., high-pressure fuel pump).'}, {'value': None, 'unit': 'whp', 'notes': 'Bolt-on mods include CAI, downpipe, headers, TB spacer, ECU remap/tune. Quality parts essential for reliability. Note: E85/E-blend requires flex fuel kit for optimal performance and protection.'}], 'fueling': {'fuel_type': 'unknown', 'injectors': None, 'pump': None, 'hpfp': None, 'port_injection': None, 'notes': None}, 'boost': {'psi': None, 'notes': None}, 'hardware': {'turbo': None, 'manifold': None, 'downpipe': None, 'intake': None, 'intercooler': None, 'exhaust': None, 'spark_plugs': None, 'cooling': [], 'drivetrain': [], 'engine_internal': [], 'notes': 'None specified.'}, 'tuning': {'tuner_or_shop': None, 'ecu_solution': 'Tu

# Resources 

- Olama Documentations - https://docs.ollama.com/capabilities/structured-outputs#python
- pydantic - https://docs.pydantic.dev/latest/api/base_model/